# Training Spike - SEA-LION v3 9B

**Purpose: de-risk the stack before Phase 3 builds 800-1500 examples.**

Base model is locked to `aisingapore/Gemma-SEA-LION-v3-9B-IT` (Phase 0, re-confirmed
2026-09-02: E2B's tokenizer advantage doesn't survive a bf16-less T4 - Unsloth forces
fp32 for gemma4, so E2B loaded *larger* than 9B in 4-bit, 7.45GB vs 6.16GB). This
notebook no longer compares models; it only verifies the stack on 9B:

1. Does it load in 4-bit on a free T4?
2. Does Unsloth take LoRA steps on it?
3. How much VRAM, how long per step?
4. Does it emit coherent, grounded **Burmese** after 30 steps?

Throwaway 18-example seed set. NOT the Phase 3 dataset.

**Runtime -> Change runtime type -> T4 GPU** before running.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install unsloth
!pip -q install --no-deps --upgrade peft accelerate bitsandbytes

In [ ]:
# Bug 2 fix (confirmed 2026-09-02): a stale compiled cache from an earlier,
# differently-configured run in this same Colab session was the "Half and
# BFloat16" dtype clash in matmul_lora during backward. Clearing it before
# unsloth is imported lets 9B complete all 30 LoRA steps cleanly. Safe to run
# even if the cache doesn't exist yet (first run of a fresh kernel).
!rm -rf /content/unsloth_compiled_cache
print("cleared unsloth_compiled_cache (if it existed)")

## Google Drive - dataset & checkpoints

**One-time manual step**: upload these two files (already generated locally, in this
repo) to a folder in your Drive, e.g. `MyDrive/eps-burmese/`:
- `data/train/phase3_train.jsonl` (~522 rows)
- `data/eval/phase4_holdout.jsonl` (~50 rows)

LoRA checkpoints will also save into this same folder so they survive a Colab
disconnect. Update `DRIVE_DIR` below if you use a different folder name.

---

**From here on the notebook is split into three sections that must run in *separate*
kernels** (Bug 3: Unsloth monkeypatches `transformers` process-wide and irreversibly, so
training and generation can never share a kernel):

1. **PHASE 4 - baseline eval** (below): fresh kernel, never import unsloth.
2. **PHASE 5 - QLoRA training**: restart runtime, re-run cells 1-2 (install + cache-clear),
   then run only that section.
3. **PHASE 6 - final eval**: restart runtime again, never import unsloth, run only that
   section.

Each section's first code cell mounts Drive again since a restart clears the mount.

## PHASE 4 - Baseline eval (base model, no adapter)

**Run this section in a fresh kernel that never imports `unsloth`.** Do not run cells 1-2
(install/cache-clear) before this - they're only needed for the training section.

Loads SEA-LION v3 9B via plain `transformers` + `bitsandbytes` (the clean-kernel path
proven working in Bug 3's investigation - Unsloth's own generation path is broken on
this stack), runs all ~50 held-out questions from `phase4_holdout.jsonl`, and saves the
outputs to Drive as the "BEFORE" side of the comparison.

In [ ]:
# Fresh kernel - install only what plain generation needs. Do NOT install/import
# unsloth in this kernel (Bug 3: its monkeypatch breaks generate() irreversibly).
!pip -q install -U transformers accelerate bitsandbytes peft

import json, time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/eps-burmese"
HOLDOUT_PATH = f"{DRIVE_DIR}/phase4_holdout.jsonl"
BASE_MODEL = "aisingapore/Gemma-SEA-LION-v3-9B-IT"

holdout = [json.loads(l) for l in open(HOLDOUT_PATH, encoding="utf-8") if l.strip()]
print(len(holdout), "held-out questions | grounded=",
      sum(r["kind"] == "grounded" for r in holdout), "refusal=",
      sum(r["kind"] == "refusal" for r in holdout))


def prompt_of(r):
    return (r["system"] + "\n\n### Context\n" + r["context"]
            + "\n\n### Question\n" + r["question"])


def msg(role, text, parts):
    return {"role": role,
            "content": [{"type": "text", "text": text}] if parts else text}


def needs_parts(tok):
    for parts in (False, True):
        try:
            tok.apply_chat_template([msg("user", "hi", parts)], tokenize=True,
                                    add_generation_prompt=True, return_tensors="pt")
            return parts
        except Exception:
            continue
    raise RuntimeError("neither content format works for this tokenizer")


def generate(model, tok, r, parts, n=300):
    text = tok.apply_chat_template([msg("user", prompt_of(r), parts)],
                                   tokenize=False, add_generation_prompt=True)
    enc = tok(text=text, return_tensors="pt", add_special_tokens=False).to("cuda")
    eos_ids = model.generation_config.eos_token_id
    eos_first = eos_ids[0] if isinstance(eos_ids, list) else eos_ids
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else eos_first
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=n, do_sample=False,
                             eos_token_id=eos_ids, pad_token_id=pad_id)
    gen_ids = out[0][enc["input_ids"].shape[1]:]
    return tok.decode(gen_ids, skip_special_tokens=True)


t0 = time.time()
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=BitsAndBytesConfig(load_in_4bit=True),
    device_map="cuda")
model.eval()
print("loaded in", round(time.time() - t0, 1), "s | VRAM",
      round(torch.cuda.memory_allocated() / 1e9, 2), "GB")
PARTS = needs_parts(tok)
print("content_parts =", PARTS)

In [ ]:
t0 = time.time()
baseline_outputs = []
for i, r in enumerate(holdout):
    out = generate(model, tok, r, PARTS)
    baseline_outputs.append({"id": r["id"], "kind": r["kind"],
                              "question": r["question"], "gold": r["answer"],
                              "output": out})
    if (i + 1) % 5 == 0 or i == len(holdout) - 1:
        print(f"[{i+1}/{len(holdout)}] {time.time()-t0:.0f}s elapsed")

OUT_PATH = f"{DRIVE_DIR}/phase4_baseline_outputs.jsonl"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    for row in baseline_outputs:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print("saved", len(baseline_outputs), "outputs ->", OUT_PATH)

# quick sanity spot-check
for row in baseline_outputs[:2]:
    print("=" * 60)
    print("Q:", row["question"])
    print("GOLD:", row["gold"][:200])
    print("MODEL:", row["output"][:200])

## PHASE 5 - QLoRA training on the real dataset

**Restart the runtime now** (Runtime -> Restart session), then re-run cells 1-2 (install
unsloth + Bug 2 cache-clear) before this section. Do not run Phase 4's cells in this
kernel - they never imported unsloth, this section does, and Bug 3 means the two can't
coexist.

Config: `r=16`, target all attention + MLP projections, `max_seq_length=2048`, 3 epochs,
`fp16` (T4 has no bf16 - confirmed in the spike), `learning_rate=1e-4` (lower than the
spike's throwaway `2e-4`, which collapsed output on a 30-step/16-example toy run - this
is the full ~522-example dataset over full epochs, a completely different regime).
Checkpoints save to Drive every 50 steps so a disconnect doesn't lose progress.

In [ ]:
import json, time, torch
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/eps-burmese"
TRAIN_PATH = f"{DRIVE_DIR}/phase3_train.jsonl"
CKPT_DIR = f"{DRIVE_DIR}/checkpoints"
BASE_MODEL = "aisingapore/Gemma-SEA-LION-v3-9B-IT"
MAX_SEQ = 2048

TRAIN = [json.loads(l) for l in open(TRAIN_PATH, encoding="utf-8") if l.strip()]
print(len(TRAIN), "training examples | grounded=",
      sum(r["kind"] == "grounded" for r in TRAIN), "refusal=",
      sum(r["kind"] == "refusal" for r in TRAIN))

In [ ]:
from unsloth import FastLanguageModel

t0 = time.time()
model, tok = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ, dtype=None, load_in_4bit=True)
print("loaded in", round(time.time() - t0, 1), "s | VRAM",
      round(torch.cuda.memory_allocated() / 1e9, 2), "GB")


def msg(role, text, parts):
    return {"role": role,
            "content": [{"type": "text", "text": text}] if parts else text}


def needs_parts(tok):
    for parts in (False, True):
        try:
            tok.apply_chat_template([msg("user", "hi", parts)], tokenize=True,
                                    add_generation_prompt=True, return_tensors="pt")
            return parts
        except Exception:
            continue
    raise RuntimeError("neither content format works for this tokenizer")


PARTS = needs_parts(tok)
print("content_parts =", PARTS)

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0.0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=0)

In [ ]:
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

BF16 = torch.cuda.is_bf16_supported()
FP16 = not BF16
print("bf16 supported:", BF16, "-> using fp16:", FP16)


def prompt_of(r):
    return (r["system"] + "\n\n### Context\n" + r["context"]
            + "\n\n### Question\n" + r["question"])


def build_text(tok, r, parts):
    return tok.apply_chat_template(
        [msg("user", prompt_of(r), parts),
         msg("assistant", r["answer"], parts)], tokenize=False)


ds = Dataset.from_list([{"text": build_text(tok, r, PARTS)} for r in TRAIN])

trainer = SFTTrainer(
    model=model, tokenizer=tok, train_dataset=ds,
    args=SFTConfig(per_device_train_batch_size=1, gradient_accumulation_steps=4,
                   warmup_steps=20, num_train_epochs=3, learning_rate=1e-4,
                   logging_steps=10, optim="adamw_8bit", weight_decay=0.01,
                   lr_scheduler_type="linear", seed=0, output_dir=CKPT_DIR,
                   save_steps=50, save_total_limit=3,
                   report_to="none", max_length=MAX_SEQ,
                   fp16=FP16, bf16=BF16,
                   dataset_text_field="text"))

print("accelerator.mixed_precision =", trainer.accelerator.mixed_precision)
print("total optimizer steps:", trainer.state.max_steps if trainer.state.max_steps else "computed at train()")

FastLanguageModel.for_training(model)
t1 = time.time()
stats = trainer.train()
print("trained in", round(time.time() - t1, 1), "s | final_loss",
      round(stats.training_loss, 4), "| peak",
      round(torch.cuda.max_memory_reserved() / 1e9, 2), "GB")

FINAL_ADAPTER = f"{CKPT_DIR}/final"
trainer.save_model(FINAL_ADAPTER)
print("saved final adapter ->", FINAL_ADAPTER)

## PHASE 6 - Final eval (fine-tuned model vs. baseline)

**Restart the runtime again**, and do **not** import `unsloth` in this kernel - same
clean-generation requirement as Phase 4. Loads the base model plain, applies the saved
LoRA adapter via `peft`, and runs the identical ~50 held-out questions so the outputs are
directly comparable to `phase4_baseline_outputs.jsonl`.

In [ ]:
# Fresh kernel - same clean-generation install as Phase 4. No unsloth here.
!pip -q install -U transformers accelerate bitsandbytes peft

import json, time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/eps-burmese"
HOLDOUT_PATH = f"{DRIVE_DIR}/phase4_holdout.jsonl"
ADAPTER_PATH = f"{DRIVE_DIR}/checkpoints/final"
BASE_MODEL = "aisingapore/Gemma-SEA-LION-v3-9B-IT"

holdout = [json.loads(l) for l in open(HOLDOUT_PATH, encoding="utf-8") if l.strip()]
print(len(holdout), "held-out questions")


def prompt_of(r):
    return (r["system"] + "\n\n### Context\n" + r["context"]
            + "\n\n### Question\n" + r["question"])


def msg(role, text, parts):
    return {"role": role,
            "content": [{"type": "text", "text": text}] if parts else text}


def needs_parts(tok):
    for parts in (False, True):
        try:
            tok.apply_chat_template([msg("user", "hi", parts)], tokenize=True,
                                    add_generation_prompt=True, return_tensors="pt")
            return parts
        except Exception:
            continue
    raise RuntimeError("neither content format works for this tokenizer")


def generate(model, tok, r, parts, n=300):
    text = tok.apply_chat_template([msg("user", prompt_of(r), parts)],
                                   tokenize=False, add_generation_prompt=True)
    enc = tok(text=text, return_tensors="pt", add_special_tokens=False).to("cuda")
    eos_ids = model.generation_config.eos_token_id
    eos_first = eos_ids[0] if isinstance(eos_ids, list) else eos_ids
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else eos_first
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=n, do_sample=False,
                             eos_token_id=eos_ids, pad_token_id=pad_id)
    gen_ids = out[0][enc["input_ids"].shape[1]:]
    return tok.decode(gen_ids, skip_special_tokens=True)


t0 = time.time()
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=BitsAndBytesConfig(load_in_4bit=True),
    device_map="cuda")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("loaded base+adapter in", round(time.time() - t0, 1), "s | VRAM",
      round(torch.cuda.memory_allocated() / 1e9, 2), "GB")
PARTS = needs_parts(tok)
print("content_parts =", PARTS)

In [ ]:
t0 = time.time()
finetuned_outputs = []
for i, r in enumerate(holdout):
    out = generate(model, tok, r, PARTS)
    finetuned_outputs.append({"id": r["id"], "kind": r["kind"],
                               "question": r["question"], "gold": r["answer"],
                               "output": out})
    if (i + 1) % 5 == 0 or i == len(holdout) - 1:
        print(f"[{i+1}/{len(holdout)}] {time.time()-t0:.0f}s elapsed")

OUT_PATH = f"{DRIVE_DIR}/phase6_finetuned_outputs.jsonl"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    for row in finetuned_outputs:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print("saved", len(finetuned_outputs), "outputs ->", OUT_PATH)

for row in finetuned_outputs[:2]:
    print("=" * 60)
    print("Q:", row["question"])
    print("GOLD:", row["gold"][:200])
    print("MODEL:", row["output"][:200])

## Compare BEFORE vs AFTER

Runs in the same Phase 6 kernel (Drive already mounted, no GPU needed for this part).
Automated checks first, then prints random pairs for **your** fluency/tone judgment - no
metric substitutes for a native reader, per the project's own rule (`SPIKE_RUN_STATE.md`).

In [ ]:
import json, re, random

baseline = {r["id"]: r for r in
            (json.loads(l) for l in open(f"{DRIVE_DIR}/phase4_baseline_outputs.jsonl", encoding="utf-8"))}
finetuned = {r["id"]: r for r in
             (json.loads(l) for l in open(f"{DRIVE_DIR}/phase6_finetuned_outputs.jsonl", encoding="utf-8"))}
ids = [i for i in baseline if i in finetuned]
print(len(ids), "matched held-out rows")

REFUSAL_MARK = "ပေးထားသော အချက်အလက်များတွင်"  # start of the fixed refusal text
HANGUL = re.compile(r"[가-힣]+")
ARTICLE_NUM = re.compile(r"Article (\d+(?:-\d+)?)")


def korean_term_retention(gold, output):
    gold_terms = set(HANGUL.findall(gold))
    if not gold_terms:
        return None
    kept = sum(1 for t in gold_terms if t in output)
    return kept / len(gold_terms)


def refused(text):
    return REFUSAL_MARK in text


def article_match(ctx_or_gold_text, output):
    gold_arts = set(ARTICLE_NUM.findall(ctx_or_gold_text))
    if not gold_arts:
        return None
    out_arts = set(ARTICLE_NUM.findall(output))
    return bool(gold_arts & out_arts)


rows = []
for i in ids:
    b, f = baseline[i], finetuned[i]
    gold = b["gold"]
    rows.append({
        "id": i, "kind": b["kind"],
        "kt_before": korean_term_retention(gold, b["output"]),
        "kt_after": korean_term_retention(gold, f["output"]),
        "refusal_correct_before": (refused(b["output"]) == (b["kind"] == "refusal")),
        "refusal_correct_after": (refused(f["output"]) == (f["kind"] == "refusal")),
        "article_match_before": article_match(gold, b["output"]),
        "article_match_after": article_match(gold, f["output"]),
    })

n = len(rows)


def avg(key):
    vals = [r[key] for r in rows if r[key] is not None]
    return sum(vals) / len(vals) if vals else float("nan")


def rate(key):
    return sum(1 for r in rows if r[key]) / n


print(f"{'metric':40s} {'BEFORE':>10s} {'AFTER':>10s}")
print(f"{'Korean-term retention (avg)':40s} {avg('kt_before'):10.1%} {avg('kt_after'):10.1%}")
print(f"{'refusal/grounded correctness':40s} {rate('refusal_correct_before'):10.1%} {rate('refusal_correct_after'):10.1%}")
print(f"{'article-number match (heuristic)':40s} "
      f"{sum(1 for r in rows if r['article_match_before']) / max(1, sum(1 for r in rows if r['article_match_before'] is not None)):10.1%} "
      f"{sum(1 for r in rows if r['article_match_after']) / max(1, sum(1 for r in rows if r['article_match_after'] is not None)):10.1%}")

print()
print("=" * 78)
print("RANDOM SAMPLE FOR YOUR REVIEW (fluency, tone, term-glossing naturalness)")
print("=" * 78)
random.seed(0)
for i in random.sample(ids, min(8, len(ids))):
    b, f = baseline[i], finetuned[i]
    print("-" * 78)
    print("Q:", b["question"], f"  [{b['kind']}]")
    print("GOLD:  ", b["gold"][:300])
    print("BEFORE:", b["output"][:300])
    print("AFTER: ", f["output"][:300])